In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

T = 20         # number of time steps
nS = 20          # number of price grid points
nY = 8          # number of y reserve levels
nG = 10          # number of gamma grid points
L = 1.0          # AMM invariant constant
gamma = 0.01     # arbitrage threshold
sigma = 0.2      # annual volatility
dt = 1/252       # time step size in years (daily)

# Grids
S_grid = np.linspace(1, 1.01, nS)
Y_grid = np.linspace(1, 1.01, nY)
gamma_grid = np.linspace(0.0001, 0.05, nG)

S_mesh, Y_mesh = np.meshgrid(S_grid, Y_grid, indexing='ij')

# Step 1: Create 3D meshgrid (nS, nY, nG)
S3, Y3, G3 = np.meshgrid(S_grid, Y_grid, gamma_grid, indexing='ij')  # All shape (nS, nY, nG)

In [2]:
def transition_prob(s0, s1, sigma, dt):
    return (1 / (s1 * sigma * np.sqrt(2 * np.pi * dt))) * \
        np.exp(- (np.log(s1) - np.log(s0))**2 / (2 * sigma**2 * dt))

transition_probs = np.zeros((nS, nS))
for i in range(nS):
    for j in range(nS):
        transition_probs[i, j] = transition_prob(S_grid[i], S_grid[j], sigma, dt)
    transition_probs[i] /= transition_probs[i].sum()  # Normalize
    

In [3]:
# %%

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm



# Compute GBM transition matrix
def transition_prob(s0, s1, sigma, dt):
    return (1 / (s1 * sigma * np.sqrt(2 * np.pi * dt))) * \
        np.exp(- (np.log(s1) - np.log(s0))**2 / (2 * sigma**2 * dt))

# Helper functions
def find_closest_index(grid, val):
    return np.argmin(np.abs(grid - val))

# Initialize value function
VV = np.zeros((T+1, nS, nY))
for j in range(nS):
    for k in range(nY):
        VV[T, j, k] = (L**2 * S_grid[j] + Y_grid[k]**2) / Y_grid[k]
progress_bar = tqdm(total=T, desc="Dynamic Programming")

# Instead of nested loops, vectorize the S and Y operations
for t in reversed(range(T)):


    # Step 2: Compute pool price and exit values
    P3 = Y3**2 / L**2

    # Initialize fee and y_next arrays
    fee3 = np.zeros_like(S3)
    y_next3 = np.zeros_like(Y3)

    # Three conditions
    cond1 = S3 > P3 / (1 - G3)
    cond2 = S3 < P3 * (1 - G3)
    cond3 = ~(cond1 | cond2)

    # Compute fee and y_next under each condition
    fee3[cond1] = G3[cond1] / (1 - G3[cond1]) * (L * np.sqrt((1 - G3[cond1]) * S3[cond1]) - Y3[cond1])
    y_next3[cond1] = L * np.sqrt((1 - G3[cond1]) * S3[cond1])

    fee3[cond2] = G3[cond2] / (1 - G3[cond2]) * (L * np.sqrt((1 - G3[cond2]) / S3[cond2]) - L**2 / Y3[cond2])
    y_next3[cond2] = L * np.sqrt(S3[cond2] / (1 - G3[cond2]))

    y_next3[cond3] = Y3[cond3]

    # Step 3: Find closest y index (l_grid) over entire 3D tensor
    # Result shape: (nS, nY, nG)
    l_grid = np.abs(Y_grid[None, None, :] - y_next3[..., None]).argmin(axis=-1)
    
    # Step 4: Compute continuation value
    # VV[t+1] shape is (nS, nY)
    continuation3 = np.zeros_like(S3)

    # We now iterate only over j in S
    for j in range(nS):
        weights = transition_probs[j]  # shape (nS,)
        # Gather V values: shape (nY, nG), indexed by l_grid
        vals = VV[t+1, :, :]  # shape (nS, nY)
        for k in range(nY):
            for g in range(nG):
                l = l_grid[j, k, g]
                continuation3[j, k, g] = np.dot(weights, VV[t+1, :, l])

    # Step 5: Total value and argmax
    total_val = fee3 + continuation3  # shape (nS, nY, nG)
    best_val = np.max(total_val, axis=2)  # shape (nS, nY)
    best_gamma_idx = np.argmax(total_val, axis=2)
    best_gamma = gamma_grid[best_gamma_idx]

    # Step 6: Choose between exit and continuation
    exit_vals = (L**2 * S_mesh + Y_mesh**2) / Y_mesh  # shape (nS, nY)
    VV[t] = np.maximum(exit_vals, best_val)


Dynamic Programming:   0%|          | 0/20 [00:00<?, ?it/s]

In [5]:
import numpy as np

nS, nY, nG = 2, 3, 2

# Example VV[t+1]
VV_next = np.array([
    [10, 20, 30],   # for S_0
    [40, 50, 60]    # for S_1
])  # shape (2, 3)

# Example l_grid: closest Y index (just mocked values between 0 and 2)
l_grid = np.array([
    [  # j = 0
        [0, 1],  # k = 0 → gamma = 0, 1
        [1, 2],  # k = 1
        [2, 0]   # k = 2
    ],
    [  # j = 1
        [2, 1],
        [0, 2],
        [1, 0]
    ]
])  # shape (2, 3, 2)


In [9]:
VV_slice = np.empty((nS, nY, nG, nS))  # j, k, g, m

for j in range(nS):
    temp = VV_next[:, l_grid[j]]  # shape (nS, nY, nG)
    VV_slice[j] = temp.transpose(1, 2, 0)  # → (nY, nG, nS)

print(VV_slice.shape)

(2, 3, 2, 2)


In [13]:
import numpy as np

# Setup
nS, nY, nG = 2, 3, 2

# Mock VV[t+1] (value function at next time step)
VV_next = np.array([
    [10, 20, 30],   # S index 0
    [40, 50, 60]    # S index 1
])  # shape (nS=2, nY=3)

# Mock l_grid: which Y index to use for each (j, k, g)
l_grid = np.array([
    [  # j = 0
        [0, 1],  # k = 0
        [1, 2],  # k = 1
        [2, 0]   # k = 2
    ],
    [  # j = 1
        [2, 1],
        [0, 2],
        [1, 0]
    ]
])  # shape (nS=2, nY=3, nG=2)

# Mock transition matrix: each row j is prob from j → m
transition_probs = np.array([
    [0.7, 0.3],  # from S0 to S0 (70%), S1 (30%)
    [0.2, 0.8],  # from S1 to S0 (20%), S1 (80%)
])  # shape (nS, nS)

# -----------------------------
# Your original triple-loop code
# -----------------------------
cont_loop = np.zeros((nS, nY, nG))

for j in range(nS):
    weights = transition_probs[j]  # shape (nS,)
    for k in range(nY):
        for g in range(nG):
            l = l_grid[j, k, g]
            cont_loop[j, k, g] = np.dot(weights, VV_next[:, l])

# -----------------------------
# Vectorized einsum version
# -----------------------------

VV_slice = np.empty((nS, nY, nG, nS))  # shape: (j, k, g, m)
for j in range(nS):
    temp = VV_next[:, l_grid[j]]  # shape: (nS, nY, nG)
    VV_slice[j] = temp.transpose(1, 2, 0)  # shape: (nY, nG, nS)
# for j in range(nS):
#     for k in range(nY):
#         for g in range(nG):
#             l = l_grid[j, k, g]  # scalar
#             VV_slice[j, k, g, :] = VV_next[:, l]  # this is the correct slice!
# print(f"VV_slice: {VV_slice.shape}")
# continuation3[j, k, g] = sum over m of weights[j, m] * VV_slice[j, k, g, m]
cont_vec = np.einsum('jm,jkgm->jkg', transition_probs, VV_slice)

# -----------------------------
# Compare
# -----------------------------

print("Original looped continuation values:")
print(cont_loop)

print("\nVectorized continuation values:")
print(cont_vec)

print("\nAre they equal?", np.allclose(cont_loop, cont_vec))


VV_slice: (2, 3, 2, 2)
Original looped continuation values:
[[[19. 29.]
  [29. 39.]
  [39. 19.]]

 [[54. 44.]
  [34. 54.]
  [44. 34.]]]

Vectorized continuation values:
[[[19. 29.]
  [29. 39.]
  [39. 19.]]

 [[54. 44.]
  [34. 54.]
  [44. 34.]]]

Are they equal? True
